### Requirements
Needs `plotly` and `ipywidgets`.

In [1]:
import os, sys
from pathlib import Path

# Directory containing this notebook.
NOTEBOOK_DIR = Path.cwd()

# Add the scripts folder to the import path, then import data_ingest.
sys.path.insert(0, str(NOTEBOOK_DIR / "scripts"))

from get_data_dict import data_ingest
from gap_report import bucket, LABELS, _count

# Raw TxSON data folder: the TXSON_DATA env var, or a default sibling path.
DATA_FOLDER = str(Path(os.environ.get(
    "TXSON_DATA",
    NOTEBOOK_DIR.parent / "datasets" / "TxSON_data_2026-02-24",
)))

In [2]:
# prewash_the_data fills missing hourly timestamps with NaN so gaps show as red blocks.
ingest = data_ingest(
    input_data_folder=DATA_FOLDER,
    prewash_the_data=True,
    clean_the_data=False,
    # download = True,
    # prewash_folder=None, # where to download prewash data. If None, it will be set to "prewashed_data" in the working directory
)
met_dict, soil_dict = ingest.get_data_dict()

print(f"Met  stations: {len(met_dict):>2}  ->  {list(met_dict)}")
print(f"Soil stations: {len(soil_dict):>2}  ->  {list(soil_dict)}")

Met  stations:  6  ->  ['CB01_met', 'CB04_met', 'CB06_met', 'FD02_met', 'FD03_met', 'WC05_met']
Soil stations: 33  ->  ['CB01', 'CB04', 'CB06', 'CB07', 'CB09', 'CB10', 'CB15', 'CB19', 'CB20', 'CB25', 'CB26', 'CB27', 'CB31', 'CB32', 'CB33', 'FD02', 'FD03', 'FD08', 'FD11', 'FD12', 'FD13', 'FD14', 'FD16', 'FD17', 'FD18', 'FD21', 'FD22', 'FD23', 'FD24', 'FD28', 'FD29', 'FD30', 'WC05']


# AI CODED VISUALIZATION:

In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from gap_report import bucket, LABELS, _count  # gap bucketing logic (single source)

# Two source dictionaries keyed by station name. "Flag" is not a measurement.
DATASETS = {"Soil": soil_dict, "Met": met_dict}
IGNORE_COLS = {"Flag"}

# Distinct line colors for overlaid variables (first one = the classic blue).
LINE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
               "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

# One color per gap-length bucket (keys match gap_report.LABELS), green -> dark red.
NA_OPACITY = 0.8  # one shading opacity for every bucket, so light colors read
GAP_COLORS = {"<24h": "#2ca02c", "1-7d": "#ff7f0e",
              "7-30d": "#d62728", ">30d": "#7f0000"}

# Most measurement columns any station has -> how many line traces to pre-create.
MAX_LINES = max([len([c for c in d.columns if c not in IGNORE_COLS])
                 for src in DATASETS.values() for d in src.values()] or [1])


def na_blocks(mask, index):
    """Group a boolean NA mask into contiguous (start, end) timestamp spans."""
    m = np.asarray(mask, dtype=bool)
    if not m.any():
        return []
    edges = np.diff(np.concatenate(([0], m.astype(np.int8), [0])))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0] - 1  # inclusive end of each run
    return [(index[s], index[e]) for s, e in zip(starts, ends)]


def _rgba(hex_color, alpha):
    """'#rrggbb' -> 'rgba(r,g,b,a)': translucent band fill with an opaque outline,
    so thin (short) gaps still show up as a crisp colored line."""
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"


# ---- figure ---------------------------------------------------------------
# MAX_LINES reusable line traces (shown/hidden per selection), then the four
# fixed NA legend swatches. _draw updates them all in place.
fig = go.FigureWidget()
for _ in range(MAX_LINES):
    fig.add_scatter(mode="lines", line=dict(width=1), connectgaps=False,
                    visible=False)
for _label in LABELS:
    fig.add_scatter(x=[None], y=[None], mode="markers", visible=False,
                    marker=dict(symbol="square", size=11,
                                color=GAP_COLORS[_label], opacity=NA_OPACITY),
                    name=f"NA {_label}")
fig.update_layout(
    template="plotly_white", height=480,
    margin=dict(l=60, r=60, t=92, b=40), hovermode="x unified",
    xaxis_title="Date", title=dict(x=0.5, xanchor="center"),
    yaxis2=dict(overlaying="y", side="right", visible=False, showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
last_fig = fig  # the live figure, handy for exporting the current view


def _draw(dataset, station, variables, df):
    """Overlay variables. A single variable gets NA gaps shaded by length. With
    'Split y-axes' on, the first variable uses the left axis and the rest use a
    right-hand axis, so variables in different units can be compared."""
    if len(df.index) > 1:
        step = pd.Timedelta(np.median(np.diff(df.index.values)))
    else:
        step = pd.Timedelta(hours=1)
    pad = step / 2
    k = len(variables)
    split = axis_cb.value and k >= 2     # right axis for variables after the first

    with fig.batch_update():
        # ---- line traces (one slot per selected variable, hide the rest) ----
        for i in range(MAX_LINES):
            tr = fig.data[i]
            if i < k:
                s = pd.to_numeric(df[variables[i]], errors="coerce")
                na_pct = 100 * s.isna().mean() if len(s) else 0.0
                on_right = split and i >= 1
                tr.x = df.index
                tr.y = s.to_numpy()
                tr.line.color = LINE_COLORS[i % len(LINE_COLORS)]
                tr.name = (f"{variables[i]} ({na_pct:.1f}% NA)"
                           + (" [right]" if on_right else ""))
                tr.yaxis = "y2" if on_right else "y"
                tr.visible = True
                tr.showlegend = k > 1        # single var: rely on title + swatches
            else:
                tr.visible = False

        # ---- right-hand axis: shown only when splitting ----
        if split:
            right_vars = variables[1:]
            fig.layout.yaxis2.visible = True
            fig.layout.yaxis2.title.text = right_vars[0] if len(right_vars) == 1 else "value"
        else:
            fig.layout.yaxis2.visible = False

        # ---- NA shading + swatches: only meaningful for a single variable ----
        if k == 1:
            s = pd.to_numeric(df[variables[0]], errors="coerce")
            blocks = na_blocks(s.isna().to_numpy(), df.index)
            spans = [(end - start) + step for start, end in blocks]
            fig.layout.shapes = [
                dict(type="rect", xref="x", yref="paper",
                     x0=start - pad, x1=end + pad, y0=0, y1=1,
                     fillcolor=_rgba(GAP_COLORS[bucket(span)], NA_OPACITY),
                     line=dict(color=GAP_COLORS[bucket(span)], width=2),
                     layer="below")
                for (start, end), span in zip(blocks, spans)
            ]
            counts = _count(spans)
            for j, label in enumerate(LABELS):
                sw = fig.data[MAX_LINES + j]
                sw.name = f"NA {label}: {counts[label]}"
                sw.visible = True
            n_na, n = int(s.isna().sum()), len(s)
            pct = 100 * n_na / n if n else 0.0
            title = (f"{dataset} · {station} · {variables[0]}   —   "
                     f"{n_na:,} NA / {n:,} ({pct:.1f}%) in {len(blocks):,} gap(s)")
            fig.layout.yaxis.title.text = variables[0]
        else:
            fig.layout.shapes = []
            for j in range(len(LABELS)):
                fig.data[MAX_LINES + j].visible = False
            shown = ", ".join(variables) if k <= 4 else f"{k} variables"
            title = f"{dataset} · {station}   —   {shown}"
            fig.layout.yaxis.title.text = variables[0] if split else "value"

        fig.layout.title.text = title


# ---- controls -------------------------------------------------------------
dataset_dd  = widgets.Dropdown(options=list(DATASETS), description="Dataset:")
station_dd  = widgets.Dropdown(description="Station:")
variable_dd = widgets.SelectMultiple(description="Variables:", rows=6)
axis_cb     = widgets.Checkbox(value=False, description="Split y-axes")
_updating   = {"on": False}  # suppress handlers during programmatic option changes


def _redraw(*_):
    df = DATASETS.get(dataset_dd.value, {}).get(station_dd.value)
    variables = [v for v in variable_dd.value if df is not None and v in df.columns]
    if df is None or not variables:
        return
    _draw(dataset_dd.value, station_dd.value, variables, df)


def _sync_variable_options():
    df = DATASETS.get(dataset_dd.value, {}).get(station_dd.value)
    cols = [c for c in df.columns if c not in IGNORE_COLS] if df is not None else []
    keep = tuple(v for v in variable_dd.value if v in cols)  # preserve where possible
    _updating["on"] = True
    variable_dd.options = cols
    variable_dd.value = keep if keep else ((cols[0],) if cols else ())
    _updating["on"] = False


def _sync_station_options():
    keys = list(DATASETS.get(dataset_dd.value, {}))
    _updating["on"] = True
    station_dd.options = keys
    station_dd.value = keys[0] if keys else None
    _updating["on"] = False


def _on_dataset_change(*_):
    if _updating["on"]:
        return
    _sync_station_options()
    _sync_variable_options()
    _redraw()


def _on_station_change(*_):
    if _updating["on"]:
        return
    _sync_variable_options()
    _redraw()


def _on_variable_change(*_):
    if _updating["on"]:
        return
    _redraw()


dataset_dd.observe(_on_dataset_change, names="value")
station_dd.observe(_on_station_change, names="value")
variable_dd.observe(_on_variable_change, names="value")
axis_cb.observe(_on_variable_change, names="value")

# Draw the first view before display so the traces are part of the figure's
# initial state.
_sync_station_options()
_sync_variable_options()
_redraw()

# Show the controls, then the figure as its own top-level output. A FigureWidget
# nested in a VBox/HBox renders blank until the first resize; shown standalone it
# draws its initial traces immediately.
display(widgets.HBox([dataset_dd, station_dd, variable_dd, axis_cb]))
display(fig)


FigureWidget({
    'data': [{'connectgaps': False,
              'line': {'color': '#1f77b4', 'width': 1},
              'mode': 'lines',
              'name': 'Ppt (2.2% NA)',
              'showlegend': False,
              'type': 'scatter',
              'uid': '5d2786d6-0962-4052-9bab-297c313560cb',
              'visible': True,
              'x': array(['2014-09-29T18:00:00.000000', '2014-09-29T19:00:00.000000',
                          '2014-09-29T20:00:00.000000', ..., '2024-03-19T06:00:00.000000',
                          '2024-03-19T07:00:00.000000', '2024-03-19T08:00:00.000000'],
                         shape=(83007,), dtype='datetime64[us]'),
              'y': {'bdata': ('exSuR+F6H0AAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA'),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'connectgaps': False,
              'line': {'width': 1},
              'mode': 'lines',
              'type': 'scatter',
              'uid': 'd7d5e691

Hold ctrl while selecting variables to overlay mutiple variables. For two variables with different units, select the "Split y-axes" option. Bin coloring disabled for mutiple variables